# TIM Example 3 — Encoder, synchronization, and advanced timer

**Target:** STM32F103C8T6 · TIM1–TIM4  
**Focus:** Select specialized timer modes and their device limits.

## Feature classification

| Mode | Timer support / input | Behavior | Example use |
|---|---|---|---|
| Encoder interface | TIM1–TIM4, CH1/CH2 | Hardware counts quadrature transitions and direction | Position/speed |
| Slave reset/gated/trigger | General/advanced timers | Reset, gate, or start count from trigger input | Synchronized period measurement |
| Master TRGO | TIM1–TIM4 | Export update/compare/enable as a trigger | ADC trigger or timer chain |
| One-pulse | Output-capable timer channel | Generate one timed pulse after trigger/start | Trigger pulse / timing window |
| Complementary PWM + dead time | TIM1 advanced-control | CH1N–CH3N, dead-time insertion, break protection | Power-stage learning only |

Encoder mode uses two timer input channels and requires correct quadrature wiring and polarity. Enable the timer clock and configure both pins as timer inputs; use the HAL encoder start function. Do not perform slow application work in capture/update callbacks.

## Encoder start example

Example shown with TIM3, where default CH1/CH2 inputs are PA6/PA7. Confirm remap and the actual package pinout before wiring.

```c
HAL_TIM_Encoder_Start(&htim3, TIM_CHANNEL_ALL);
uint16_t position = (uint16_t)__HAL_TIM_GET_COUNTER(&htim3);
// Convert count delta to direction/speed in application code.
```


## Master/slave synchronization

CubeMX configures a master timer's TRGO source (for example, update) and a slave timer's trigger input/slave mode. This is useful for phase-related waveforms or deterministic peripheral triggers. Validate that the selected ITR/trigger route exists in RM0008; timer interconnects are not arbitrary.

## TIM1 advanced-control variations

TIM1 adds complementary CH1N–CH3N outputs, programmable dead time, break input, and repetition counter. These features are not present on TIM2–TIM4. Default main channels are PA8–PA11 and complementary outputs are generally PB13–PB15; break is mapped on PB12 by default. Check remapping and package pin availability.

The repository's `STM32F103C8T6_Basic_Timer_IT` is not a complementary-output exercise: it drives TIM1_CH4 (PA11), with no CH4N output. Its break input is disabled and dead time is zero. Trace its update-DMA sequence in PWM Example 3, then use this section only as an extension.

> **Power-stage safety:** Never connect complementary timer outputs directly to a high-current bridge. Confirm dead time, break polarity, inactive-state behavior, gate-driver logic, current limiting, and safe power-off states on the actual hardware. Begin with a logic analyzer/oscilloscope and a low-voltage isolated setup.

## Feature selection checklist

- Choose a general-purpose timer for time base, capture, compare, encoder, and basic PWM.
- Choose TIM1 only when advanced-control functions are required.
- Check timer clock domain, 16-bit range, channel pin mapping, trigger routing, and peripheral conflicts.
- Distinguish timer update interrupt from capture/compare interrupts; each needs the matching HAL start/API and IRQ handler.
- ADC trigger synchronization is separate from PWM output: select a valid timer TRGO route supported by ADC1/ADC2 on the F103.